## Imports

In [36]:
from pydantic import BaseModel, Field
from typing import Optional, List
from ollama import chat
import pandas as pd
import tiktoken
import argparse
import json
import os
import re
import hashlib
import csv

## Get Human Annotated Ground Truth, CTMAP, and Manuscript

In [9]:
ds_name = "adipose_Emont2022"
folder_name = f"../data/{ds_name}"
evidence_human =f"{folder_name}/evidence_human/evidence.json"
ctmap = f"{folder_name}/ctmap/ctmap.json"
manuscript = f"{folder_name}/manuscript/manuscript.txt"

In [27]:
def load_evidence(fn, ds_name):
    df = pd.read_json(fn)
    # Unwrap the 'source' column (contains a dictionary) into separate columns
    source_df = df['source'].apply(pd.Series)
    extracted_df = df['extracted'].apply(pd.Series).add_prefix('extracted_')
    # derived_df = df['derived'].apply(pd.Series).add_prefix('derived_')
    df = pd.concat([df.drop(columns=['source', 'extracted']), source_df, extracted_df], axis=1)
    df["ds"] = ds_name
    del df["derived"]
    return df

def short_hash(s: str) -> str:
    """Generate a 7-character hash from a given string."""
    return hashlib.sha256(s.encode()).hexdigest()[:7]

## Run Chunking Code

#### adapted from llm/chunk_filter_merge.ipynb

In [10]:
def split_text_into_windows(corpus, window_size=500, overlap=250, document_name="doc"):
    results = []
    step_size = window_size - overlap

    for start_idx in range(0, len(corpus), step_size):
        end_idx = min(start_idx + window_size, len(corpus))

        extracted_text = corpus[start_idx:end_idx]

        results.append({
            "extracted_text": extracted_text,
            "start_idx": start_idx,
            "end_idx": end_idx,
            "index_type": "character",
            "document_name": document_name
        })

        if end_idx == len(corpus):
            break

    return results

def tokenize_with_offsets(text, encoding="cl100k_base"):
    """Tokenizes text and returns tokens with their character start positions."""
    enc = tiktoken.get_encoding(encoding)
    tokens = []

    etks = enc.encode(text)
    dtks, ptks = enc.decode_with_offsets(etks)

    assert len(etks) == len(ptks)

    for t, p in zip(etks, ptks):
        token = enc.decode_single_token_bytes(t).decode("utf-8")
        tobj = {
            "token": token,
            "enc_token": t,
            "start_idx": p,
            "end_idx": p + len(token),
        }
        tokens.append(tobj)
    return tokens

def split_text_into_windows(corpus, window_size=200, overlap=50, document_name="doc", encoding="cl100k_base"):
    tokens = tokenize_with_offsets(corpus, encoding=encoding)
    results = []

    step_size = window_size - overlap
    for start_idx in range(0, len(tokens), step_size):
        end_idx = min(start_idx + window_size, len(tokens))

        window_tokens = tokens[start_idx:end_idx]

        extracted_text = corpus[window_tokens[0]['start_idx']:window_tokens[-1]['end_idx']]

        results.append({
            "extracted_text": extracted_text,
            "start_idx": window_tokens[0]['start_idx'],
            "end_idx": window_tokens[-1]['end_idx'],
            "index_type": "character",
            "document_name": document_name
        })

        if end_idx == len(tokens):
            break

    return results

In [11]:
with open(manuscript, "r") as f:
    paper = f.read().strip()

# Split the text into windows
windows = split_text_into_windows(paper, window_size=100, overlap=10, document_name=ds_name)

In [13]:
windows[-1]

{'extracted_text': ' covariate when only a single condition was present. \n \nReporting summary \n \nFurther information on research design is available in the Nature Research Reporting Summary linked to this paper.',
 'start_idx': 66138,
 'end_idx': 66329,
 'index_type': 'character',
 'document_name': 'adipose_Emont2022'}

## Run LLM Against the Chunks (T/F)

In [19]:
# Permissive class object (allows for missing fields)
class GeneFinder(BaseModel):
    hasCelltypeGene: bool = Field(False, description="Does the text discuss biological cell type marker genes?")

In [17]:
def extract_genes(user_content, system_prompt, data_model, model="llama3.2"):
    response = chat(
        messages=[
            {
                'role': 'system',
                'content': system_prompt,
            },
            {
                'role': 'user',
                'content': user_content,
            }
        ],
        model = model,
        format = data_model.model_json_schema(),
        options={'temperature': 0},  # Make responses more deterministic

    )
    genes = data_model.model_validate_json(response.message.content)
    return genes.model_dump()

system_prompt = """
You are an expert biologist specialized in identifying discussions about cell types and their associated marker genes in scientific texts. Your job is to determine if the provided text discusses relationships between cell types and their marker genes. The text may contain scientific jargon, abbreviations, and complex sentences. Your task is to identify whether the text discusses cell types and their marker genes.

Does the following text discuss relationships between cell types and their marker genes?

Criteria for “True”:
- Mentions or implies cell types AND specifically mentions a gene or set of genes.
- Discusses marker genes as identifying, characterizing, or differentiating particular cell types.

Otherwise, return “False”.
"""

In [20]:
check = []
for idx, window in enumerate(windows):
    genes = extract_genes(window["extracted_text"], system_prompt, GeneFinder, model="llama3.2")
    if idx % 50 == 0:
        print(f"Processed {idx} rows.")
    window.update({"hasCelltypeGene": genes["hasCelltypeGene"]})
    check.append(window)

Processed 0 rows.
Processed 50 rows.
Processed 100 rows.
Processed 150 rows.


## Filter out False Chunks

In [21]:
filt = [i for i in check if i["hasCelltypeGene"]]
len(filt)

86

## Merge Adjacent Chunks

In [22]:
merged = []

current = filt[0]

for next_row in filt[1:]:
    if current['end_idx'] > next_row['start_idx']:
        # Calculate the non-overlapping part
        overlap_len = current['end_idx'] - next_row['start_idx']
        current['extracted_text'] += next_row['extracted_text'][overlap_len:]
        current['end_idx'] = next_row['end_idx']
    else:
        merged.append(current)
        current = next_row.copy()

merged.append(current)

In [23]:
merged[2]

{'extracted_text': ' of adipocytes and ASPCs did not differ between depots, but depot clearly affects the distribution of cells within these populations (Extended Data Fig. 1a, b, Extended Data Table 2, Supplementary Fig. 1). In our limited cohort, we could not detect major effects of body mass index (BMI) on cell-type proportions. To assess this finding at larger scale, we used our dataset as a reference to estimate cell-type proportions in bulk RNA-sequencing data12 obtained from',
 'start_idx': 1987,
 'end_idx': 2453,
 'index_type': 'character',
 'document_name': 'adipose_Emont2022',
 'hasCelltypeGene': True}

In [24]:
def reformat_merged(merged):
    reformatted = []
    for m in merged:
        reformatted.append({
            "extracted": {
                "organism": "",
                "cell_type_label": "",
                "cell_source": "",
                "cell_state": "",
                "feature_name": "",
                "feature_type": "",
            },
            "derived": {
                "organism": "",
                "cell_type_id": "",
                "cell_type_label": "",
                "cell_source": "",
                "cell_state": "",
                "feature_name": "",
                "feature_type": "",
                "feature_identifier": "",
                "feature_identifier_type": "",
            },
            "source": {
                "source_rationale": m["extracted_text"],
                "source_type": "text",
                "source_id": "text"
            }
        })
    return reformatted

In [25]:
final_set = reformat_merged(merged)

## Extract CTMGs with LLM

In [30]:
# Permissive class object (allows for missing fields)
class MarkerGene(BaseModel):
    marker_gene_name: Optional[str] = Field(
        None, 
        description="The name of the marker gene that is associated with a specific cell type."
    )
    cell_type_name: Optional[str] = Field(
        None, 
        description="The name of the cell type for which this gene serves as a marker."
    )

class MarkerGeneList(BaseModel):
    genes: List[MarkerGene]  # A list of marker genes extracted from the input text

# Restrictive class object does not allow for missing fields
class MarkerGeneStrict(BaseModel):
    marker_gene_name: str = Field(
        ..., 
        description="The name of the marker gene that is associated with a specific cell type."
    )
    cell_type_name: str = Field(
        ..., 
        description="The name of the cell type for which this gene serves as a marker."
    )

class MarkerGeneListStrict(BaseModel):
    genes: List[MarkerGeneStrict]  # A list of marker genes extracted from the input text

In [31]:
MODEL_TO_USE = "llama3.2"

LLMODELS = {
    "deepseek-r1": "deepseek-r1",
    "llama3.2": "llama3.2",
    "llama3.3": "llama3.3"
}

DATA_MODELS = {
    "MarkerGeneListStrict": MarkerGeneListStrict,
    "MarkerGeneStrict": MarkerGeneStrict
}

system_prompt = """
You are an expert in genomics analyzing scientific literature to extract marker genes for different cell types. 

Your goal is to identify and structure marker gene data from the given text. For each marker gene mentioned, extract:
- The **gene name** (marker_gene_name).
- The **cell type** it is associated with (cell_type_name).

The data must be extracted as written in the text, without any modifications.

Return the results in **structured JSON format** with the following schema:
{
    "genes": [
        {
            "cell_type_name": "Neuron",
            "marker_gene_name": "GeneX",
        },
        ...
    ]
}
"""

model_metadata = {
        "model_id" : LLMODELS[MODEL_TO_USE],
        "system_prompt": system_prompt,
        "system_prompt_hash": short_hash(system_prompt),
        "data_model": DATA_MODELS["MarkerGeneListStrict"].__name__
}

In [32]:
def extract_genes(user_prompt, system_prompt, data_model, model=MODEL_TO_USE):
    response = chat(
        messages=[
            {
                'role': 'system',
                'content': system_prompt,
            },
            {
                'role': 'user',
                'content': user_prompt,
            }
        ],
        model = model,
        format = data_model.model_json_schema(),
        options={'temperature': 0},  # Make responses more deterministic

    )
    genes = data_model.model_validate_json(response.message.content)
    return genes.model_dump()["genes"]

In [33]:
df = load_evidence(evidence_human, ds_name)

In [34]:
tdf = df.query("source_type == 'text'").copy()
uniq_src = tdf.source_rationale.unique()
ct_mgs = []

for idx, text in enumerate(uniq_src):
    print(f"{idx + 1}/{len(uniq_src)}")
    ct_mg = extract_genes(text, model_metadata["system_prompt"], DATA_MODELS[model_metadata["data_model"]], model=model_metadata["model_id"])
    
    if len(ct_mg) == 0:
        continue
        
    for g in ct_mg:
        tmp = {
            "extracted": {
                "organism": "homo_sapiens",
                "cell_type_label": g["cell_type_name"],
                "cell_source": None,
                 "cell_state": None,
                "feature_name": g["marker_gene_name"],
                "feature_type": "gene",
            },
            "derived": {
                "organism": "homo_sapiens",
                "cell_type_id": None,
                "cell_type_label": g["cell_type_name"],
                "cell_source": None,
                "cell_state": None,
                "feature_name": g["marker_gene_name"],
                "feature_type": "gene",
                "feature_identifier": None,
                "feature_identifier_type": None,
            },
            "source": {
                "source_type": "text",
                "source_rationale": text,
                "source_id": "text"
            }
        }

        ct_mgs.append(tmp)

1/9
2/9
3/9
4/9
5/9
6/9
7/9
8/9
9/9


In [35]:
tmp

{'extracted': {'organism': 'homo_sapiens',
  'cell_type_label': 'Adipocyte',
  'cell_source': None,
  'cell_state': None,
  'feature_name': 'PPARG',
  'feature_type': 'gene'},
 'derived': {'organism': 'homo_sapiens',
  'cell_type_id': None,
  'cell_type_label': 'Adipocyte',
  'cell_source': None,
  'cell_state': None,
  'feature_name': 'PPARG',
  'feature_type': 'gene',
  'feature_identifier': None,
  'feature_identifier_type': None},
 'source': {'source_type': 'text',
  'source_rationale': 'One such gene is PPARG, which is highly expressed in all adipocytes (Extended Data Fig. 14a).',
  'source_id': 'text'}}

In [43]:
llm_folder = os.path.join(folder_name, f"workflow_evidence_llm_{model_metadata['model_id']}_{model_metadata['data_model']}_{model_metadata['system_prompt_hash']}")
os.mkdir(llm_folder)

output_path = os.path.join(llm_folder, "evidence.json") 
with open(output_path, 'w') as f:
    json.dump(ct_mgs, f, indent = 4)

model_fn = os.path.join(llm_folder, "model_metadata.json")
with open(model_fn, 'w') as f:
    json.dump(model_metadata, f, indent = 4)

## Map Feature Names to Feature Identifiers

In [44]:
def update_json_with_identifiers(json_path, tsv_path, output_path):
    # Step 1: Load the feature mapping, uppercasing keys
    feature_map = {}
    with open(tsv_path, 'r') as tsvfile:
        reader = csv.reader(tsvfile, delimiter='\t')
        for row in reader:
            if len(row) >= 2:
                key = row[0].strip().upper()
                value = row[1].strip()
                feature_map[key] = value

    # Step 2: Load JSON
    with open(json_path, 'r') as f:
        data = json.load(f)

    # Step 3: Process and conditionally update
    for entry in data:
        derived = entry.get("derived", {})
        
        organism = derived.get("organism", "").strip().lower()
        if derived.get("feature_name", "") is str:
            feature_name = derived.get("feature_name", "").strip().upper()
        else:
            feature_name = derived.get("feature_name", "")
        if organism == "homo_sapiens" and feature_name in feature_map:
            derived["feature_identifier"] = feature_map[feature_name]
            derived["feature_identifier_type"] = "ensembl"

    # Step 4: Save updated JSON
    with open(output_path, 'w') as f:
        json.dump(data, f, indent=2)

In [45]:
gmap = "../data/gmap.txt"

update_json_with_identifiers(output_path, gmap, output_path)

## Map Cell Type Labels to Cell Type Identifiers

In [47]:
def find_key_given_value(label_map: dict, label):
    for key in label_map.keys():
        if label in label_map[key]:
            return key
    return None

def update_json(label_map, json_fn = 'evidence.json'):
    with open(json_fn, 'r') as file:
        data = json.load(file)
        for obj in data:
            if obj['derived']['cell_type_label'] is not None:
                obj['derived']['cell_type_id'] = find_key_given_value(label_map, obj['derived']['cell_type_label'].upper().strip())
        with open(json_fn, "w") as file:
            json.dump(data, file, indent = 4)
    
    print("Processing complete!")

In [48]:
with open(ctmap, 'r') as f:
    ctmap_json = json.load(f)

update_json(ctmap_json, output_path)

Processing complete!
